In [1]:
!pip install sentencepiece sacrebleu evaluate datasets
!pip install -U transformers==4.44.2 peft==0.10.0 accelerate==0.33.0 bitsandbytes

In [2]:
from peft import LoraConfig, get_peft_model
import os
from datasets import load_dataset
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer
import evaluate
from transformers import TrainerCallback
import time
import os

os.environ["WANDB_DISABLED"] = "true"
os.environ["WANDB_MODE"] = "offline"


2025-11-03 05:47:22.908883: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1762148842.935366    8659 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1762148842.943747    8659 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [ ]:
dataset = load_dataset("csv", data_files="/kaggle/input/d/muaadhnazly/sin-eng/sin-eng.csv")

dataset = dataset["train"].train_test_split(test_size=0.2)
train_dataset = dataset["train"]
eval_dataset = dataset["test"]

tokenizer = AutoTokenizer.from_pretrained("facebook/nllb-200-distilled-600M") 
model = AutoModelForSeq2SeqLM.from_pretrained( "facebook/nllb-200-distilled-600M", 
        torch_dtype=torch.float32, 
        device_map="cuda")
model.to('cuda')


In [ ]:
source_lang = "sin_Sinh"
target_lang = "eng_Latn"
tokenizer.src_lang = source_lang
tokenizer.tgt_lang = target_lang

max_length = 128
def preprocess(examples):
    # Get raw sentences
    inputs = examples["sinhala"]
    targets = examples["english"]
    
    # Tokenize source and target
    model_inputs = tokenizer(
        inputs,
        max_length=max_length,
        padding="max_length",
        truncation=True
    )
    
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(
            targets,
            max_length=max_length,
            padding="max_length",
            truncation=True
        )
    
    # Put labels in model_inputs
    model_inputs["labels"] = labels["input_ids"]
    
    return model_inputs
# def preprocess(examples):
#     inputs = examples["sinhala"]
#     targets = examples["english"]
#     model_inputs = tokenizer(inputs, max_length=max_length, truncation=True)
#     labels = tokenizer(targets, max_length=max_length, truncation=True)
#     model_inputs["labels"] = labels["input_ids"]
#     return model_inputs

train_dataset = train_dataset.map(preprocess, batched=True)
eval_dataset = eval_dataset.map(preprocess, batched=True)


lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type="SEQ_2_SEQ_LM"
)

model = get_peft_model(model, lora_config)


In [ ]:

training_args = Seq2SeqTrainingArguments(
    output_dir="./nllb_sinhala2english",
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    learning_rate=5e-4,
    num_train_epochs=20,
    logging_strategy="steps",
    eval_strategy="steps",
    save_strategy="steps",
    logging_dir="./logs",
    predict_with_generate=True,
    fp16=True,
    gradient_accumulation_steps=2,      
    dataloader_pin_memory=True,
    report_to="none",                   
    load_best_model_at_end=True,
    metric_for_best_model="loss",      
    save_total_limit=3,    
)

# Load BLEU metric
bleu = evaluate.load("sacrebleu")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    result = bleu.compute(predictions=decoded_preds, references=[[l] for l in decoded_labels])
    result["bleu"] = result["score"]
    return result

In [ ]:
class RealTimeProgressCallback(TrainerCallback):
    def on_step_end(self, args, state, control, **kwargs):
        logs = kwargs.get("logs", {})
        if "loss" in logs:
            current_time = time.strftime("%H:%M:%S", time.localtime())
            print(f"[{current_time}] Step: {state.global_step}, Loss: {logs['loss']:.4f}", flush=True)
        if state.global_step % 3880 == 0:
            trainer.save_model(f"./nllb_sinhala2english_gpu/checkpoint-{state.global_step}")


trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)
# Add the callback to the trainer
trainer.add_callback(RealTimeProgressCallback)

trainer.args.logging_steps = 1 
trainer.args.eval_steps = 3880
trainer.train()

model.save_pretrained("./nllb_sinhala2english_lora")
tokenizer.save_pretrained("./nllb_sinhala2english_lora")

In [ ]:
import os
import json
from transformers import AutoTokenizer

save_dir = "./nllb_sinhala2english_lora"

# Create dir if missing
os.makedirs(save_dir, exist_ok=True)

# --- Save model + tokenizer ---
trainer.model.save_pretrained(save_dir)
trainer.tokenizer.save_pretrained(save_dir)

def serialize_training_args(args):
    data = {}
    for k, v in vars(args).items():
        try:
            json.dumps(v)  # test serializability
            data[k] = v
        except TypeError:
            data[k] = str(v)
    return data

training_args_dict = serialize_training_args(trainer.args)

with open(f"{save_dir}/training_args.json", "w") as f:
    json.dump(training_args_dict, f, indent=4)

print(f"✅ Model, tokenizer, and training args saved in: {save_dir}")

In [ ]:
!zip -r "/kaggle/working/nllb_sinhala2english_lora.zip" "/kaggle/working//nllb_sinhala2english_lora"

In [ ]:
import json
import os
import shutil

# Paths
WORKING_DIR = "/kaggle/working"
OUTPUT_DIR = os.path.join(WORKING_DIR, "finetuned")
ZIP_PATH = os.path.join(WORKING_DIR, "nllb_sinhala2english_lora.zip")

# Make sure the output folder exists
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Move the zip into the output folder
shutil.copy(ZIP_PATH, OUTPUT_DIR)

# Dataset metadata
metadata = {
    "title": "nllb-fine-tuned-600",
    "id": "MuaadhNazly/nllb-fine-tuned-600",
    "licenses": [{"name": "CC0-1.0"}]
}

# Save metadata JSON inside the output folder
metadata_path = os.path.join(OUTPUT_DIR, "dataset-metadata.json")
with open(metadata_path, "w") as f:
    json.dump(metadata, f)

# Install/upgrade Kaggle API
!pip install kaggle --upgrade

# Set up Kaggle API key
os.makedirs("/root/.config/kaggle", exist_ok=True)
!cp "/kaggle/input/api-key-2/kaggle.json" "/root/.config/kaggle/"
!chmod 600 /root/.config/kaggle/kaggle.json

# Create the Kaggle dataset
!kaggle datasets create -p {OUTPUT_DIR} -u --quiet
